In [7]:
import jax
import numpyro
import numpy as np
from jax import numpy as jnp
import pandas as pd
import pickle as pkl
import pixelpop
import h5py


/home/sofia.alvarez/.conda/envs/gwjax311/lib/python3.11/site-packages/h5py/__init__.py:36: UserWarning: h5py is running against HDF5 1.14.3 when it was built against 1.14.2, this may cause problems
  _warn(("h5py is running against HDF5 {0} when it was built against {1}, "


In [8]:
varcut = 1
mmin = 3

In [9]:
with open('data/posteriors_chieff.pkl', 'rb') as ff:
    _posteriors = pkl.load(ff)

with open('data/injection_set_chieff.pkl', 'rb') as ff:
    injections = pkl.load(ff)


In [10]:
def clean_data(data, min_m=mmin, max_m=200, max_z=2.3, remove=False):
    pixelpop.utils.data.clean_par(data, 'log_mass_1', jnp.log(min_m), jnp.log(max_m), remove=remove)
    pixelpop.utils.data.clean_par(data, 'redshift', 0., max_z, remove=remove)
    
keys = ['mass_1', 'mass_ratio', 'a_1', 'a_2', 'cos_tilt_1', 'cos_tilt_2', 'redshift', 'prior']
#posteriors = {k: jnp.array([p[k] for p in _posteriors]) for k in keys}

print(f"I have {_posteriors['mass_1'].shape[0]} events")
    
# convert to log m1, log m2 space

posteriors = pixelpop.utils.data.convert_m1_to_lm1(_posteriors)
injections = pixelpop.utils.data.convert_m1_to_lm1(injections)

clean_data(posteriors)
clean_data(injections, remove=True)

I have 69 events


In [11]:
parameters = ['log_mass_1', 'redshift'] # parameters of "PixelPop"-ed space
other_parameters = ['mass_ratio', 'a', 't'] # nuisance parameters, we define parametric population model

# Pin these parametric hyperparameters to set values
priors = {'max_z': [[2.3], numpyro.distributions.Delta], 'qmin': [[0.1], numpyro.distributions.Delta]}

In [12]:
probabilistic_model, initial_value = pixelpop.models.probabilistic.setup_probabilistic_model(
    posteriors, # individual GW parameters
    injections, # injections to estimate selection effects
    parameters, # parameters to infer with PixelPop ICAR model
    other_parameters, # nuisance parameters
    100, # number of bins along each axis
    minima={'log_mass_1': np.log(mmin)}, # minimum of space
    maxima={}, # maxima set to default values
    priors=priors, # priors which differ from defaults
    UncertaintyCut=np.sqrt(varcut), # convergence criteria for likelihood estimator
    parametric_models={}, # parametric models for nuisance parameters are set to defaults
    length_scales=False, # same ICAR Gaussian coupling strength in all directions
    random_initialization=True, # initialize ICAR model from random Gaussian draw
    lower_triangular=False, # full space is allowed
    )

dimension = 2, density = 100
dimension = 2, density = 100
Using default mass_ratio model SimplePowerlaw_MassRatio
Using default a model iid_normal_spin
Using default t model tilt_iid
Using default prior beta = Uniform(-2, 7) in mass_ratio model
Using custom prior qmin = Delta(0.1) in mass_ratio model
Using default prior mu_spin = Uniform(0, 1) in a model
Using default prior var_spin = Uniform(0.005, 0.25) in a model
Using default prior mu_tilt = Uniform(-1, 1) in t model
Using default prior sig_tilt = Uniform(0.1, 4) in t model
Using default prior zeta_tilt = Uniform(0, 1) in t model


In [37]:
output = pixelpop.models.probabilistic.inference_loop(
    probabilistic_model, 
    model_kwargs={'posteriors': posteriors, 'injections': injections}, 
    initial_value=initial_value,
    warmup=50, 
    tot_samples=50,
    thinning=1,
    pacc=0.65,
    maxtreedepth=5,
    num_samples=4,
    parallel=1,
    run_dir='../../results/',
    name=f'm1z_varcut{varcut}',
    print_keys=['Nexp', 'log_likelihood', 'log_likelihood_variance', 'lnsigma', 'beta'], 
    # ^ for checking on chains while they run
    dense_mass=False
    )

Warming up chain #1 out of 1


warmup: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:16<00:00,  2.97it/s, 31 steps of size 2.29e-01. acc. prob=0.60]


drawing thinned samples:   9%|█████████████                                                                                                                                  | 1/11 [00:05<00:50,  5.09s/it]



                                                mean       std    median      5.0%     95.0%     n_eff     r_hat
                                      Nexp     28.89      1.24     29.24     26.99     30.09    -49.50      0.87
                                      beta      1.36      0.36      1.40      0.88      1.76     14.14      0.87
                                   lnsigma      1.29      0.03      1.29      1.27      1.32      6.35      0.87
                            log_likelihood    203.51      4.42    203.85    198.11    208.23      8.12      0.87
                   log_likelihood_variance      1.11      0.04      1.11      1.06      1.17    -23.40      0.87
  worst n_eff: merger_rate_density[95, 31]     -0.89      0.39     -0.94     -1.34     -0.33 -11185.24      0.87
                         worst r_hat: Nexp     28.89      1.24     29.24     26.99     30.09    -49.50      0.87




drawing thinned samples:  18%|██████████████████████████                                                                                                                     | 2/11 [00:09<00:39,  4.42s/it]

...



                                                mean       std    median      5.0%     95.0%     n_eff     r_hat
                                      Nexp     30.06      2.14     29.70     26.90     32.06     21.02      0.97
                                      beta      1.92      0.49      1.88      1.22      2.78      7.51      1.30
                                   lnsigma      1.19      0.06      1.17      1.13      1.31      3.38      1.88
                            log_likelihood    226.60     13.79    231.02    200.95    242.21      3.74      1.68
                   log_likelihood_variance      1.10      0.07      1.10      0.98      1.20     43.92      0.99
   worst n_eff: merger_rate_density[2, 67]     -2.49      2.97     -2.98     -6.41      1.39      2.69      3.22
  worst r_hat: merger_rate_density[35, 75]      0.77      2.99      1.60     -2.87      4.29      2.93      4.94




drawing thinned samples:  91%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████             | 10/11 [00:41<00:04,  4.01s/it]



                                                mean       std    median      5.0%     95.0%     n_eff     r_hat
                                      Nexp     30.28      2.40     29.70     26.90     32.65     24.39      0.98
                                      beta      1.95      0.48      2.02      1.22      2.62      6.80      1.36
                                   lnsigma      1.18      0.06      1.16      1.12      1.27      3.51      1.79
                            log_likelihood    227.93     13.85    232.28    206.75    244.53      3.92      1.59
                   log_likelihood_variance      1.10      0.07      1.10      1.02      1.20     46.48      0.98
  worst n_eff: merger_rate_density[82, 64]      0.25      0.97      0.33     -0.88      1.48      2.71      3.67
  worst r_hat: merger_rate_density[15, 30]     -1.83      2.23     -1.97     -4.71      0.78      2.92      4.11




drawing thinned samples: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 11/11 [00:44<00:00,  4.09s/it]



                                                mean       std    median      5.0%     95.0%     n_eff     r_hat
                                      Nexp     30.25      2.36     29.70     26.90     32.65     27.77      0.99
                                      beta      1.99      0.53      2.02      1.22      2.78      7.01      1.26
                                   lnsigma      1.18      0.06      1.15      1.11      1.27      3.64      1.80
                            log_likelihood    228.40     13.36    232.28    206.75    244.53      4.13      1.46
                   log_likelihood_variance      1.10      0.08      1.10      0.97      1.20     71.12      0.98
  worst n_eff: merger_rate_density[61, 19]     -0.11      1.54     -0.03     -1.93      1.97      2.72      3.93
  worst r_hat: merger_rate_density[92, 62]     -0.43      1.84     -0.47     -2.63      1.87      2.81      4.08



In [68]:
path = '/home/sofia.alvarez/public_html/voronoi_population/gw_nonparametric/pixelpop-realistic-astropop/results/PE/rate_pp_paper_3dcorr_CEpopbins_45-45-45par_lnmqxz_nonparameteric_lnmqx_unccut_2.0/seed_4_w100000_s100000_t100_p1_acc0p80_mcmc_meta_data.pkl'

In [69]:
with open(path, 'rb') as ff:
    meta = pkl.load(ff)

In [70]:
neffs = [meta[key]['n_eff'] for key in meta]

In [75]:
rhats = [meta[key]['r_hat'] for key in meta]

In [78]:
np.max(neffs)

1229.7369160450385

In [82]:
np.argmax(rhats)

57186

In [85]:
output[0][0]['merger_rate_density'].ndim

3